In [1]:
import pickle

# run the model
with open('../rf_model.pkl', 'rb') as file:
    rf_model = pickle.load(file)



In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error,r2_score

In [3]:
rf_input = pd.read_csv("../x.csv")
rf_input_clean = rf_input.drop(columns=['Unnamed: 0'], errors='ignore')



In [4]:
y_data = pd.read_csv("../y.csv")
y_pred = rf_model.predict(rf_input_clean)
y_true = y_data['total_score']
no_noise_mse = mean_squared_error(y_true, y_pred)
print("MSE with original Random Forest model:", no_noise_mse)

MSE with original Random Forest model: 3.497120120109635


In [5]:
r2_no_noise = r2_score(y_true, y_pred)
print("R² after adding noise on highly relevant variables:", r2_no_noise)

R² after adding noise on highly relevant variables: 0.8904293786569714


## Noise experiment on highly relevent variables

In [6]:
print("sadornot distinct values:", rf_input['sadornot'].unique())
print("experience distinct values:", rf_input['experience'].unique())
print("gpa_all distinct values:", rf_input['gpa_all'].unique())

sadornot distinct values: [1. 2.]
experience distinct values: [2 3 1 4 5]
gpa_all distinct values: [3.505 3.029 3.474 3.705 3.667 3.245 3.293 3.373 3.476 3.947 3.719 3.826
 2.815 3.79  3.625 2.4   3.519]


In [7]:
# add noise to variables: sadornot
# Randomly add 1, subtract 1, or make no change for each sample.
noisy_input = rf_input.copy()

sadornot_noise = np.random.choice([-1, 0, 1], size=noisy_input.shape[0])

noisy_input['sadornot'] = noisy_input['sadornot'] + sadornot_noise
noisy_input['sadornot'] = noisy_input['sadornot'].clip(lower=1, upper=2)

print(noisy_input['sadornot'].value_counts().sort_index())

sadornot
1.0    4667
2.0    4336
Name: count, dtype: int64


In [8]:
# add noise to variable: experience
# apply normal distribution to choose the reasonable noise to add on the original experience column

experience_noise = np.random.normal(loc=0, scale=1, size=noisy_input.shape[0])

noisy_input['experience'] = noisy_input['experience'] + experience_noise
noisy_input['experience'] = noisy_input['experience'].round().astype(int)
noisy_input['experience'] = noisy_input['experience'].clip(lower=1, upper=5)

print(noisy_input['experience'].head(10))

0    3
1    3
2    2
3    3
4    2
5    3
6    1
7    2
8    2
9    3
Name: experience, dtype: int64


In [9]:
# add noise to variable: gpa_all
# gpa_all is continuous numerical variable
# apply normal distribution to choose the reasonable noise to add on the original sleep_hours column

gpa_noise = np.random.normal(0, 0.05, size=noisy_input.shape[0])
noisy_input['gpa_all'] = noisy_input['gpa_all'] + gpa_noise
print(noisy_input['gpa_all'].head(10))

0    3.642287
1    3.508990
2    3.543117
3    3.516578
4    3.511738
5    3.452163
6    3.481401
7    3.548376
8    3.576166
9    3.520872
Name: gpa_all, dtype: float64


In [10]:
# re-run the xgboost model
noisy_input = noisy_input.drop(columns=['Unnamed: 0'], errors='ignore')

y_pred_noisy = rf_model.predict(noisy_input)

In [11]:
y_data = pd.read_csv("../y.csv")

y_true = y_data['total_score']
mse_noisy = mean_squared_error(y_true, y_pred_noisy)

print("MSE after adding noise on highly relevant variables:", mse_noisy)


MSE after adding noise on highly relevant variables: 7.883953351225145


In [12]:
r2_noisy = r2_score(y_true, y_pred_noisy)
print("R² after adding noise on highly relevant variables:", r2_noisy)

R² after adding noise on highly relevant variables: 0.7529825577435099


## Noise experiment on less relevent variables

In [13]:
print("has_negative_text distinct values:", rf_input['has_negative_text'].unique())
print("schedule distinct values:", rf_input['schedule'].unique())
print("have distinct values:", rf_input['have'].unique())

has_negative_text distinct values: [1. 0.]
schedule distinct values: [1. 2.]
have distinct values: [1. 2.]


In [14]:
less_relevent_noisy_input = rf_input.copy()

binary_features = {
    'has_negative_text': (0, 1),
    'schedule': (1, 2),
    'have': (1, 2)
}

for feature, (min_val, max_val) in binary_features.items():
    noise = np.random.choice([-1, 0, 1], size=less_relevent_noisy_input.shape[0])
    less_relevent_noisy_input[feature] = less_relevent_noisy_input[feature] + noise
    less_relevent_noisy_input[feature] = less_relevent_noisy_input[feature].clip(lower=min_val, upper=max_val)

print(less_relevent_noisy_input[list(binary_features.keys())].head(10))

   has_negative_text  schedule  have
0                0.0       1.0   2.0
1                1.0       1.0   1.0
2                1.0       2.0   1.0
3                0.0       2.0   1.0
4                1.0       2.0   1.0
5                1.0       1.0   2.0
6                1.0       2.0   2.0
7                0.0       2.0   1.0
8                1.0       2.0   1.0
9                0.0       1.0   1.0


In [15]:
less_relevent_noisy_input = less_relevent_noisy_input.drop(columns=['Unnamed: 0'], errors='ignore')

less_relevent_y_pred_noisy = rf_model.predict(less_relevent_noisy_input)

In [16]:
less_relevent_mse_noisy = mean_squared_error(y_true, less_relevent_y_pred_noisy)

print("MSE after adding noise for less relevant variables:", less_relevent_mse_noisy)

MSE after adding noise for less relevant variables: 3.497233576069229


In [17]:
less_relevent_r2_noisy = r2_score(y_true, less_relevent_y_pred_noisy)
print("R² after adding noise on less relevant variables:", less_relevent_r2_noisy)

R² after adding noise on less relevant variables: 0.8904258238920332


| Condition (Random Forest Model) | MSE    | RMSE (√MSE) | R²    |
|:---------------------------------|:-------|:--------------------------|:------|
| No Noise                         | 3.4971 | 1.8707                    | 0.8904 |
| Noise on Top-3 Important Features | 7.8840 | 2.8071                    | 0.7520 |
| Noise on Bottom-3 Important Features | 3.4972 | 1.8707                    | 0.8904 |

